# Búsqueda híbrida con ChromaDB

Notebook mínimo sobre `data/json/noticias_uniovi.json`.

Crea una colección Chroma con dos índices:

- `#embedding`: búsqueda semántica densa.
- `sparse_embedding`: búsqueda léxica BM25.

Después combina ambos rankings con `Rrf`, igual que en el ejemplo de la documentación de Chroma.

Nota: este ejemplo usa la Search API de Chroma (`Search`, `Knn`, `Rrf`). Si tu instalación local de Chroma todavía no soporta `collection.search(...)`, ejecútalo con una versión/entorno de Chroma que tenga esa API habilitada.

In [1]:
# Dependencias para ejecutar en Colab. No modifica el proyecto.
%pip install -q chromadb sentence-transformers snowballstemmer pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 99.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [6]:
import json
import math
import shutil
from pathlib import Path

import chromadb
import pandas as pd
from chromadb import K, Knn, Rrf, Schema, Search, SparseVectorIndexConfig, VectorIndexConfig
from chromadb.utils.embedding_functions import ChromaBm25EmbeddingFunction, SentenceTransformerEmbeddingFunction


DATA_PATH = Path("./noticias_uniovi.json")

with DATA_PATH.open("r", encoding="utf-8") as f:
    noticias = json.load(f)

df = pd.DataFrame(noticias)
display(df[["fecha", "titulo", "etiquetas", "fuente_html"]].head())

,fecha,titulo,etiquetas,fuente_html
0,22/07/2026,La nueva Junta Rectora de CRUE se reúne con el...,[Información institucional],15310136.html
1,20/07/2026,La Universidad de Oviedo reconstruye los oríge...,[Investigación],15288681.html
2,20/07/2026,La Universidá Asturiana de Branu incorpora la ...,"[Cultura, Estudiantes]",15289547.html
3,17/07/2026,El grupo Llabor de la Universidad de Oviedo re...,"[Investigación, Cultura]",15271733.html
4,16/07/2026,"El 81,43% del estudiantado aprueba la PAU en l...","[Nota de prensa, Información institucional, Es...",15260773.html


In [7]:
def clean(value) -> str:
    if isinstance(value, list):
        return ", ".join(map(str, value))
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ""
    return str(value)


def chunk_words(text: str, size: int = 160, overlap: int = 40) -> list[str]:
    words = text.split()
    if len(words) <= size:
        return [text]
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, len(words), step) if words[i:i + size]]


documents, metadatas, ids = [], [], []

for doc_id, row in df.iterrows():
    text = "\n".join([
        f"Título: {clean(row.get('titulo'))}",
        f"Fecha: {clean(row.get('fecha'))}",
        f"Etiquetas: {clean(row.get('etiquetas'))}",
        f"Resumen: {clean(row.get('resumen'))}",
        f"Noticia: {clean(row.get('noticia'))}",
    ])

    for chunk_id, chunk in enumerate(chunk_words(text)):
        ids.append(f"noticia-{doc_id}-chunk-{chunk_id}")
        documents.append(chunk)
        metadatas.append({
            "doc_id": int(doc_id),
            "chunk_id": int(chunk_id),
            "titulo": clean(row.get("titulo")),
            "fecha": clean(row.get("fecha")),
            "etiquetas": clean(row.get("etiquetas")),
            "fuente_html": clean(row.get("fuente_html")),
        })

print(f"Noticias: {len(df)}")
print(f"Chunks: {len(documents)}")

Noticias: 303
Chunks: 2039


## Hybrid search with ChromaDB (only works with chorma cloud client) - Commented -

### Create schema, idexes and collection

In [12]:
CHROMA_PATH = "chroma_uniovi_demo"
COLLECTION_NAME = "noticias_uniovi"

shutil.rmtree(CHROMA_PATH, ignore_errors=True)

dense_ef = SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

sparse_ef = ChromaBm25EmbeddingFunction(
    k=1.2,
    b=0.75,
    avg_doc_length=160.0,
    token_max_length=40,
)

schema = Schema()
schema.create_index(
    config=VectorIndexConfig(
        source_key=K.DOCUMENT,
        embedding_function=dense_ef,
        space="cosine",
    )
)
# (Only with CloudClient)
'''
schema.create_index(
    key="sparse_embedding",
    config=SparseVectorIndexConfig(
        source_key=K.DOCUMENT,
        embedding_function=sparse_ef,
        bm25=True,
    ),
)
'''
client = chromadb.PersistentClient(path=CHROMA_PATH) # CloudClient needs api key
collection = client.create_collection(
    name=COLLECTION_NAME,
    schema=schema,
)

for start in range(0, len(documents), 100):
    end = start + 100
    collection.add(
        ids=ids[start:end],
        documents=documents[start:end],
        metadatas=metadatas[start:end],
    )

print("Registros en Chroma:", collection.count())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Registros en Chroma: 2039


### Test query

In [ ]:
query = "Noticias relacionadas con la investigadora Noelia Rico Pachón"

# Dense semantic embeddings
dense_rank = Knn(
    query=query,
    key="#embedding",
    return_rank=True,
    limit=200,
)

# Sparse keyword embeddings (Only with CloudClient)
'''
sparse_rank = Knn(
    query=query,
    key="sparse_embedding",
    return_rank=True,
    limit=200,
)
'''

# Combine with RRF (Only with CloudClient)
'''
hybrid_rank = Rrf(
    ranks=[dense_rank, sparse_rank],
    weights=[0.7, 0.3],
    k=60,
)
'''

results = collection.query(query_texts=[query], n_results=10)

results

{'ids': [['noticia-35-chunk-2',
   'noticia-24-chunk-2',
   'noticia-30-chunk-0',
   'noticia-112-chunk-8',
   'noticia-215-chunk-4',
   'noticia-104-chunk-0',
   'noticia-143-chunk-6',
   'noticia-211-chunk-4',
   'noticia-12-chunk-2',
   'noticia-222-chunk-6']],
 'embeddings': None,
 'documents': [['y experiencias entre investigadores noveles y consolidados mediante un espacio de debate científico interdisciplinar e intergeneracional, articulado en torno a sesiones de micropresentaciones y pósteres. El acto inaugural ha contado con la participación del rector de la Universidad de Oviedo, Ignacio Villaverde; la directora general de Universidad del Gobierno del Principado de Asturias, Cristina González Morán; la vicerrectora de Investigación, Irene Díaz Rodríguez, y la directora de la Escuela Internacional de Doctorado, Susana Montes Rodríguez. Durante su intervención, el rector ha destacado que “el doctorado constituye uno de los pilares fundamentales de la universidad, porque es en e

In [ ]:
rows = results.rows()[0]

display(pd.DataFrame([{
    "score": row["score"],
    "fecha": row.get("fecha"),
    "titulo": row.get("titulo"),
    "fuente_html": row.get("fuente_html"),
    "texto": row["document"][:350] + "...",
} for row in rows]))

## Hybrid search with CromaDB and rank-bm25 library


In [13]:
%pip install -q rank-bm25

In [14]:
import re
import unicodedata

from rank_bm25 import BM25Okapi


STOPWORDS = set("""
a al algo algunas algunos ante antes como con contra cual cuando de del desde donde durante e el ella ellas ellos en entre era
eran es esa esas ese eso esos esta estaba estaban estamos estan estar estas este esto estos fue fueron ha han hasta hay la las le
les lo los mas me mi mis mucha muchas mucho muchos muy no nos o para pero por que se si sin sobre son su sus tambien te tiene
tienen un una unas uno unos y ya universidad oviedo
""".split())


def normalize_for_bm25(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def tokenize_for_bm25(text: str) -> list[str]:
    tokens = re.findall(r"[a-z0-9]+", normalize_for_bm25(text))
    return [token for token in tokens if len(token) > 2 and token not in STOPWORDS]


bm25 = BM25Okapi([tokenize_for_bm25(doc) for doc in documents])
id_to_pos = {item_id: pos for pos, item_id in enumerate(ids)}

print("BM25 listo sobre", len(documents), "chunks")

BM25 listo sobre 2039 chunks


In [15]:
def semantic_search(query: str, candidates_k: int = 50) -> list[dict]:
    result = collection.query(
        query_texts=[query],
        n_results=candidates_k,
        include=["documents", "metadatas", "distances"],
    )

    hits = []
    for rank, item_id in enumerate(result["ids"][0], start=1):
        pos = id_to_pos[item_id]
        hits.append({
            "id": item_id,
            "pos": pos,
            "rank_dense": rank,
            "distance": result["distances"][0][rank - 1],
        })
    return hits


def keyword_search(query: str, candidates_k: int = 50) -> list[dict]:
    scores = bm25.get_scores(tokenize_for_bm25(query))
    order = sorted(range(len(scores)), key=lambda pos: scores[pos], reverse=True)[:candidates_k]

    return [{
        "id": ids[pos],
        "pos": pos,
        "rank_bm25": rank,
        "bm25_score": float(scores[pos]),
    } for rank, pos in enumerate(order, start=1)]


def hybrid_search(query: str, top_k: int = 10, candidates_k: int = 50, dense_weight: float = 0.7, bm25_weight: float = 0.3, rrf_k: int = 60) -> pd.DataFrame:
    dense_hits = semantic_search(query, candidates_k=candidates_k)
    keyword_hits = keyword_search(query, candidates_k=candidates_k)

    fused = {}

    for hit in dense_hits:
        item = fused.setdefault(hit["id"], {
            "pos": hit["pos"],
            "rank_dense": None,
            "rank_bm25": None,
            "distance": None,
            "bm25_score": None,
            "score_rrf": 0.0,
        })
        item["rank_dense"] = hit["rank_dense"]
        item["distance"] = hit["distance"]
        item["score_rrf"] += dense_weight / (rrf_k + hit["rank_dense"])

    for hit in keyword_hits:
        item = fused.setdefault(hit["id"], {
            "pos": hit["pos"],
            "rank_dense": None,
            "rank_bm25": None,
            "distance": None,
            "bm25_score": None,
            "score_rrf": 0.0,
        })
        item["rank_bm25"] = hit["rank_bm25"]
        item["bm25_score"] = hit["bm25_score"]
        item["score_rrf"] += bm25_weight / (rrf_k + hit["rank_bm25"])

    rows = []
    for item in sorted(fused.values(), key=lambda value: value["score_rrf"], reverse=True)[:top_k]:
        meta = metadatas[item["pos"]]
        rows.append({
            "score_rrf": round(item["score_rrf"], 5),
            "rank_dense": item["rank_dense"],
            "rank_bm25": item["rank_bm25"],
            "bm25_score": None if item["bm25_score"] is None else round(item["bm25_score"], 3),
            "distance": None if item["distance"] is None else round(item["distance"], 4),
            "fecha": meta["fecha"],
            "titulo": meta["titulo"],
            "fuente_html": meta["fuente_html"],
            "texto": documents[item["pos"]][:350] + "...",
        })

    return pd.DataFrame(rows)

In [26]:
query = "¿Qué noticias son las más relevantes en cuanto a inteligencia artificial?"

hybrid_results = hybrid_search(
    query,
    top_k=10,
    candidates_k=50,
    dense_weight=0.7,
    bm25_weight=0.3,
)

display(hybrid_results)

,score_rrf,rank_dense,rank_bm25,bm25_score,distance,fecha,titulo,fuente_html,texto
0,0.01517,3,14,7.793,0.3840,22/05/2026,El grupo 9 de Universidades aborda el impacto ...,14605405.html,Título: El grupo 9 de Universidades aborda el ...
1,0.01505,4,13,7.826,0.3925,20/02/2026,España e India refuerzan la cooperación univer...,13808234.html,"de la inteligencia artificial, la importancia ..."
2,0.01462,2,30,6.960,0.3705,07/05/2026,Una egresada de la Universidad de Oviedo acced...,14475521.html,"alto valor añadido”. De cara al futuro, su int..."
3,0.01448,1,40,5.795,0.2876,05/11/2025,La actividad emprendedora continúa creciendo e...,12525835.html,"de las iniciativas emprendedoras, especialment..."
4,0.01414,9,15,7.729,0.4326,05/06/2026,El uso de la inteligencia artificial generativ...,14711612.html,Título: El uso de la inteligencia artificial g...
5,0.01381,13,11,7.892,0.4482,10/07/2026,La Universidad de Oviedo reúne a especialistas...,15214301.html,Título: La Universidad de Oviedo reúne a espec...
6,0.01338,6,48,5.489,0.4108,10/07/2026,La Universidad de Oviedo reúne a especialistas...,15214301.html,la responsabilidad de interpretar las grandes ...
7,0.01305,11,34,6.219,0.4402,02/03/2026,El G-9 registra 9.673 inscripciones en el XIII...,13884089.html,coordinada de la inteligencia artificial ha si...
8,0.01286,15,25,7.335,0.4516,02/12/2025,El profesor Roger Campione presenta en la Univ...,12754908.html,liderado por el catedrático de Filosofía del d...
9,0.01235,21,21,7.572,0.4841,06/07/2026,Gijón reunirá a expertos internacionales en in...,15178623.html,Título: Gijón reunirá a expertos internacional...
